# Recover Missing YOLO In-Domain Evaluations

This notebook **does not retrain any model**. It loads existing `best.pt` files from the cross-dataset runs, evaluates them on their original dataset, and creates loader-compatible `results_summary.csv` files.

Recovered evaluations:
- DAWN seeds 123 and 456 from `dawn_to_acdc_yolo11s_seed...`
- ACDC seeds 123 and 456 from `acdc_to_dawn_yolo11s_seed...`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.9 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from ultralytics import YOLO

print("Imports ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Imports ready.


In [10]:
PROJECT_ROOT = Path('/content/drive/MyDrive/Dissertation')
RUNS_DIR = PROJECT_ROOT / 'Runs' / 'yolo'

DAWN_DATA_YAML = PROJECT_ROOT / 'Datasets' / 'processed' / 'dawn_yolo' / 'dataset.yaml'
ACDC_DATA_YAML = PROJECT_ROOT / 'Datasets' / 'processed' / 'acdc_yolo' / 'acdc.yaml'

SEEDS_TO_RECOVER = [123, 456]

CONFIG = {
    'DAWN': {
        'data_yaml': DAWN_DATA_YAML,
        'source_run_template': 'dawn_to_acdc_yolo11s_seed{seed}',
        'output_run_template': 'dawn_in_domain_yolo11s_seed{seed}',
    },
    'ACDC': {
        'data_yaml': ACDC_DATA_YAML,
        'source_run_template': 'acdc_to_dawn_yolo11s_seed{seed}',
        'output_run_template': 'acdc_in_domain_yolo11s_seed{seed}',
    },
}

print('RUNS_DIR:', RUNS_DIR)
print('DAWN YAML:', DAWN_DATA_YAML)
print('ACDC YAML:', ACDC_DATA_YAML)

RUNS_DIR: /content/drive/MyDrive/Dissertation/Runs/yolo
DAWN YAML: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml
ACDC YAML: /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml


In [12]:
import torch


YOLO_ROOT = PROJECT_ROOT / "Datasets/processed/acdc_yolo"
RUNS_ROOT = PROJECT_ROOT / "Runs/yolo"

print("YOLO_ROOT:", YOLO_ROOT)
print("Exists:", YOLO_ROOT.exists())
print("RUNS_ROOT:", RUNS_ROOT)
print("Exists:", RUNS_ROOT.exists())

YOLO_ROOT: /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo
Exists: True
RUNS_ROOT: /content/drive/MyDrive/Dissertation/Runs/yolo
Exists: True


Sanity check

In [ ]:
missing_items = []

for dataset_name, cfg in CONFIG.items():
    if not cfg['data_yaml'].exists():
        missing_items.append(f"Missing YAML: {cfg['data_yaml']}")

    for seed in SEEDS_TO_RECOVER:
        source_run = RUNS_DIR / cfg['source_run_template'].format(seed=seed)
        weights_path = source_run / 'weights' / 'best.pt'
        if not weights_path.exists():
            missing_items.append(f"Missing weights: {weights_path}")

if missing_items:
    print("\n".join(missing_items))
    raise FileNotFoundError("Fix the paths listed above before continuing.")

print("All YAML files and weights were found.")

All YAML files and weights were found.


## Evaluation function

Each recovered result is saved as `results_summary.csv`, matching the analysis loader.


In [ ]:
def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def evaluate_in_domain(dataset_name, seed, batch=16, imgsz=640, device=0, workers=2):
    cfg = CONFIG[dataset_name]

    source_run_dir = RUNS_DIR / cfg['source_run_template'].format(seed=seed)
    weights_path = source_run_dir / 'weights' / 'best.pt'
    data_yaml = cfg['data_yaml']

    output_run_dir = RUNS_DIR / cfg['output_run_template'].format(seed=seed)
    output_run_dir.mkdir(parents=True, exist_ok=True)

    print('=' * 80)
    print(f'{dataset_name} in-domain evaluation | seed {seed}')
    print('Weights:', weights_path)
    print('Dataset:', data_yaml)
    print('Output:', output_run_dir)
    print('=' * 80)

    model = YOLO(str(weights_path))

    metrics = model.val(
        data=str(data_yaml),
        split='test',
        imgsz=imgsz,
        batch=batch,
        device=device,
        workers=workers,
        plots=False,
        verbose=True,
        project=str(output_run_dir),
        name='global_evaluation',
        exist_ok=True,
    )

    precision = safe_float(metrics.box.mp)
    recall = safe_float(metrics.box.mr)

    f1_score = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )

    speed = getattr(metrics, 'speed', {}) or {}
    preprocess_ms = safe_float(speed.get('preprocess'))
    inference_ms = safe_float(speed.get('inference'))
    postprocess_ms = safe_float(speed.get('postprocess'))
    fps = 1000.0 / inference_ms if np.isfinite(inference_ms) and inference_ms > 0 else np.nan

    row = {
        'dataset': dataset_name,
        'experiment': 'in_domain',
        'scenario': 'in_domain',
        'direction': 'in_domain',
        'model': 'YOLOv11s',
        'seed': seed,
        'condition': 'global',
        'mAP50-95': safe_float(metrics.box.map),
        'mAP50': safe_float(metrics.box.map50),
        'mAP75': safe_float(metrics.box.map75),
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1_score,
        'Preprocess_ms_per_image': preprocess_ms,
        'Inference_ms_per_image': inference_ms,
        'Postprocess_ms_per_image': postprocess_ms,
        'FPS': fps,
        'weights_path': str(weights_path),
        'data_yaml': str(data_yaml),
    }

    result_df = pd.DataFrame([row])
    csv_path = output_run_dir / 'results_summary.csv'
    result_df.to_csv(csv_path, index=False)

    metadata_path = output_run_dir / 'recovery_metadata.json'
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump({
            'dataset': dataset_name,
            'seed': seed,
            'source_weights': str(weights_path),
            'evaluation_yaml': str(data_yaml),
            'output_csv': str(csv_path),
            'note': 'Recovered in-domain evaluation from existing cross-dataset training weights; no retraining performed.'
        }, f, indent=2)

    print('\nSaved:', csv_path)
    display(result_df)
    return result_df

##  Run the four missing evaluations

This evaluates:
- DAWN seed 123
- DAWN seed 456
- ACDC seed 123
- ACDC seed 456


In [ ]:
all_recovered_results = []

for dataset_name in ['DAWN', 'ACDC']:
    for seed in SEEDS_TO_RECOVER:
        result = evaluate_in_domain(
            dataset_name=dataset_name,
            seed=seed,
            batch=16,
            imgsz=640,
            device=0,
            workers=2,
        )
        all_recovered_results.append(result)

recovered_df = pd.concat(all_recovered_results, ignore_index=True)
display(recovered_df)

DAWN in-domain evaluation | seed 123
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed123/weights/best.pt
Dataset: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml
Output: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed123
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.1±0.7 ms, read: 0.6±0.7 MB/s, size: 225.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 21.7Mit/s 0.0s
                 Class     Images  Instances  

,dataset,experiment,scenario,direction,model,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,FPS,weights_path,data_yaml
0,DAWN,in_domain,in_domain,in_domain,YOLOv11s,123,global,0.35225,0.552245,0.389594,0.707678,0.479959,0.571987,0.39733,4.553643,0.758463,219.604407,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...


DAWN in-domain evaluation | seed 456
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/weights/best.pt
Dataset: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml
Output: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed456
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.2 ms, read: 23.0±14.7 MB/s, size: 68.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 45.2Mit/s 0.0s
                 Class     Images  Instances 

,dataset,experiment,scenario,direction,model,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,FPS,weights_path,data_yaml
0,DAWN,in_domain,in_domain,in_domain,YOLOv11s,456,global,0.275106,0.508195,0.227976,0.634303,0.381149,0.47617,0.408286,1.021892,0.958651,978.57702,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...


ACDC in-domain evaluation | seed 123
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed123/weights/best.pt
Dataset: /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml
Output: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed123
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.1 ms, read: 0.8±0.3 MB/s, size: 437.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 161.8Mit/s 0.0s
                 Class     Images  Instances    

,dataset,experiment,scenario,direction,model,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,FPS,weights_path,data_yaml
0,ACDC,in_domain,in_domain,in_domain,YOLOv11s,123,global,0.356363,0.548418,0.366478,0.738876,0.491172,0.590083,0.353292,0.986033,0.730213,1014.164715,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...


ACDC in-domain evaluation | seed 456
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed456/weights/best.pt
Dataset: /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml
Output: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed456
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.0 ms, read: 209.8±55.1 MB/s, size: 455.8 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 161.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 7.0it/s 4.9s
                   all        540       3358      0.524      0.343      0.

,dataset,experiment,scenario,direction,model,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,FPS,weights_path,data_yaml
0,ACDC,in_domain,in_domain,in_domain,YOLOv11s,456,global,0.206947,0.36418,0.200342,0.524486,0.342873,0.414666,0.344567,0.844261,0.782485,1184.467868,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...


,dataset,experiment,scenario,direction,model,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,FPS,weights_path,data_yaml
0,DAWN,in_domain,in_domain,in_domain,YOLOv11s,123,global,0.352250,0.552245,0.389594,0.707678,0.479959,0.571987,0.397330,4.553643,0.758463,219.604407,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...
1,DAWN,in_domain,in_domain,in_domain,YOLOv11s,456,global,0.275106,0.508195,0.227976,0.634303,0.381149,0.476170,0.408286,1.021892,0.958651,978.577020,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...
2,ACDC,in_domain,in_domain,in_domain,YOLOv11s,123,global,0.356363,0.548418,0.366478,0.738876,0.491172,0.590083,0.353292,0.986033,0.730213,1014.164715,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...
3,ACDC,in_domain,in_domain,in_domain,YOLOv11s,456,global,0.206947,0.364180,0.200342,0.524486,0.342873,0.414666,0.344567,0.844261,0.782485,1184.467868,/content/drive/MyDrive/Dissertation/Runs/yolo/...,/content/drive/MyDrive/Dissertation/Datasets/p...


## Verify generated files

In [ ]:
generated_files = []

for dataset_name, cfg in CONFIG.items():
    for seed in SEEDS_TO_RECOVER:
        output_run_dir = RUNS_DIR / cfg['output_run_template'].format(seed=seed)
        csv_path = output_run_dir / 'results_summary.csv'
        generated_files.append({
            'dataset': dataset_name,
            'seed': seed,
            'csv_exists': csv_path.exists(),
            'csv_path': str(csv_path),
        })

verification_df = pd.DataFrame(generated_files)
display(verification_df)

assert verification_df['csv_exists'].all(), "At least one results_summary.csv is missing."
print("All recovery CSV files were created successfully.")

,dataset,seed,csv_exists,csv_path
0,DAWN,123,True,/content/drive/MyDrive/Dissertation/Runs/yolo/...
1,DAWN,456,True,/content/drive/MyDrive/Dissertation/Runs/yolo/...
2,ACDC,123,True,/content/drive/MyDrive/Dissertation/Runs/yolo/...
3,ACDC,456,True,/content/drive/MyDrive/Dissertation/Runs/yolo/...


All recovery CSV files were created successfully.


## Weather specific

### ACDC

In [15]:
# ============================================================
# RECOVER ACDC IN-DOMAIN RESULTS: seeds 123 and 456
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import pandas as pd

eval_configs = {
    "global": YOLO_ROOT / "acdc.yaml",
    "fog": YOLO_ROOT / "fog_only.yaml",
    "rain": YOLO_ROOT / "rain_only.yaml",
    "snow": YOLO_ROOT / "snow_only.yaml",
}

ACDC_SEEDS = [123, 456]

for seed in ACDC_SEEDS:

    print("\n" + "=" * 80)
    print(f"RECOVERING ACDC IN-DOMAIN — SEED {seed}")
    print("=" * 80)

    weights_path = (
        RUNS_ROOT
        / f"acdc_to_dawn_yolo11s_seed{seed}"
        / "weights"
        / "best.pt"
    )

    output_dir = (
        RUNS_ROOT
        / f"acdc_in_domain_yolo11s_seed{seed}"
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    assert weights_path.exists(), (
        f"Missing weights for seed {seed}:\n{weights_path}"
    )

    for condition, yaml_path in eval_configs.items():
        assert yaml_path.exists(), (
            f"Missing YAML for {condition}:\n{yaml_path}"
        )

    print("Weights:", weights_path)
    print("Output :", output_dir)

    model = YOLO(str(weights_path))

    results_rows = []

    for condition, yaml_path in eval_configs.items():

        print(f"\nEvaluating ACDC {condition.upper()} — seed {seed}...")

        metrics = model.val(
            data=str(yaml_path),
            split="test",
            imgsz=640,
            batch=16,
            device=0,
            workers=2,
            plots=False,
            project=str(output_dir / "evaluation"),
            name=f"test_{condition}",
            exist_ok=True,
            verbose=True
        )

        precision = float(metrics.box.mp)
        recall = float(metrics.box.mr)
        map50 = float(metrics.box.map50)
        map75 = float(metrics.box.map75)
        map5095 = float(metrics.box.map)

        f1 = 2 * precision * recall / (precision + recall + 1e-9)

        inference_ms = float(metrics.speed.get("inference", 0))
        fps = 1000 / inference_ms if inference_ms > 0 else None

        results_rows.append({
            "dataset": "ACDC",
            "model": "YOLOv11s",
            "experiment": "in_domain",
            "seed": seed,
            "condition": condition,
            "mAP50-95": map5095,
            "mAP50": map50,
            "mAP75": map75,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "Inference_ms_per_image": inference_ms,
            "FPS": fps
        })

    results_df = pd.DataFrame(results_rows)

    output_csv = (
        output_dir
        / "acdc_yolo11s_in_domain_results_summary.csv"
    )

    results_df.to_csv(output_csv, index=False)

    print("\nSaved successfully:")
    print(output_csv)

    display(results_df)


RECOVERING ACDC IN-DOMAIN — SEED 123
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed123/weights/best.pt
Output : /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed123

Evaluating ACDC GLOBAL — seed 123...
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.0 ms, read: 198.1±48.5 MB/s, size: 444.5 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 161.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 6.4it/s 5.4s
                   all        540       3358      0.739      0.491      0.548      0.356
                person       

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Inference_ms_per_image,FPS
0,ACDC,YOLOv11s,in_domain,123,global,0.356363,0.548418,0.366478,0.738876,0.491172,0.590083,0.846315,1181.593755
1,ACDC,YOLOv11s,in_domain,123,fog,0.447830,0.666048,0.466978,0.814496,0.602718,0.692784,0.971950,1028.860017
2,ACDC,YOLOv11s,in_domain,123,rain,0.321870,0.508605,0.321331,0.707358,0.463154,0.559782,0.891321,1121.930196
3,ACDC,YOLOv11s,in_domain,123,snow,0.387695,0.554470,0.410591,0.698773,0.500007,0.582912,0.957964,1043.880585



RECOVERING ACDC IN-DOMAIN — SEED 456
Weights: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed456/weights/best.pt
Output : /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed456

Evaluating ACDC GLOBAL — seed 456...
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 191.6±65.2 MB/s, size: 478.7 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 174.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 5.4it/s 6.2s
                   all        540       3358      0.524      0.343      0.364      0.207
                person       

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Inference_ms_per_image,FPS
0,ACDC,YOLOv11s,in_domain,456,global,0.206947,0.364180,0.200342,0.524486,0.342873,0.414666,0.848933,1177.949757
1,ACDC,YOLOv11s,in_domain,456,fog,0.301932,0.479680,0.336257,0.811058,0.409838,0.544522,0.950638,1051.925272
2,ACDC,YOLOv11s,in_domain,456,rain,0.190498,0.339837,0.178483,0.477412,0.337422,0.395392,0.880033,1136.321188
3,ACDC,YOLOv11s,in_domain,456,snow,0.206604,0.362599,0.200593,0.544782,0.317115,0.400879,0.965228,1036.024799


## DAWN

In [17]:
print((PROJECT_ROOT / "Datasets/processed/dawn_yolo").exists())

True


In [21]:
eval_configs = {
    "global": YOLO_ROOT / "dataset.yaml",
    "fog": YOLO_ROOT / "fog_only.yaml",
    "rain": YOLO_ROOT / "rain_only.yaml",
    "snow": YOLO_ROOT / "snow_only.yaml",
}

def extract_metrics(metrics_obj):
    return {
        "map50": float(metrics_obj.box.map50),
        "map50_95": float(metrics_obj.box.map),
        "mp": float(metrics_obj.box.mp),
        "mr": float(metrics_obj.box.mr),
    }

for seed in [123, 456]:

    print("=" * 80)
    print(f"Recovering DAWN in-domain seed {seed}")
    print("=" * 80)

    # Same model trained on DAWN, reused for DAWN in-domain evaluation
    weights_path = (
        RUNS_ROOT
        / f"dawn_to_acdc_yolo11s_seed{seed}"
        / "weights"
        / "best.pt"
    )

    print("Loading:", weights_path)

    model = YOLO(str(weights_path))

    results_summary = {}

    for condition, yaml_path in eval_configs.items():

        print(f"\nEvaluating: {condition}")

        metrics = model.val(
            data=str(yaml_path),
            split="test",
            imgsz=640,
            batch=16,
            plots=False,
            verbose=False
        )

        results_summary[condition] = extract_metrics(metrics)

    df = pd.DataFrame(results_summary)

    output_folder = (
        RUNS_ROOT
        / f"dawn_in_domain_yolo11s_seed{seed}"
    )

    output_folder.mkdir(parents=True, exist_ok=True)

    output_csv = output_folder / "weather_results_summary.csv"

    df.to_csv(output_csv, index=False)

    print(f"\nSaved: {output_csv}")
    display(df)

Recovering DAWN in-domain seed 123
Loading: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed123/weights/best.pt

Evaluating: global
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.9 ms, read: 64.2±50.8 MB/s, size: 122.8 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 39.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 9.3it/s 1.0s
                   all        140        920      0.708       0.48      0.552      0.352
Speed: 0.5ms preprocess, 1.1ms inference, 0.0ms loss, 1.0ms postprocess per image

Evaluating: fog
Ultralytics 8.4.99 🚀 Python-3.12.13 torch

,global,fog,rain,snow
map50,0.552245,0.597530,0.469903,0.641021
map50_95,0.352250,0.372268,0.286862,0.413445
mp,0.707678,0.748364,0.390667,0.630780
mr,0.479959,0.523389,0.416600,0.746612


Recovering DAWN in-domain seed 456
Loading: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_to_acdc_yolo11s_seed456/weights/best.pt

Evaluating: global
Ultralytics 8.4.99 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 79.6±49.7 MB/s, size: 160.7 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 48.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 9.2it/s 1.0s
                   all        140        920      0.634      0.381      0.508      0.275
Speed: 0.5ms preprocess, 1.1ms inference, 0.0ms loss, 1.1ms postprocess per image

Evaluating: fog
Ultralytics 8.4.99 🚀 Python-3.12.13 torch

,global,fog,rain,snow
map50,0.508195,0.501777,0.469478,0.593639
map50_95,0.275106,0.270584,0.249115,0.396076
mp,0.634303,0.634720,0.727449,0.724908
mr,0.381149,0.348090,0.419633,0.426265


In [22]:
for seed in [123, 456]:
    path = (
        RUNS_ROOT
        / f"dawn_in_domain_yolo11s_seed{seed}"
        / "weather_results_summary.csv"
    )

    print(f"\nSeed {seed}: {path.exists()}")
    display(pd.read_csv(path))


Seed 123: True


,global,fog,rain,snow
0,0.552245,0.597530,0.469903,0.641021
1,0.352250,0.372268,0.286862,0.413445
2,0.707678,0.748364,0.390667,0.630780
3,0.479959,0.523389,0.416600,0.746612



Seed 456: True


,global,fog,rain,snow
0,0.508195,0.501777,0.469478,0.593639
1,0.275106,0.270584,0.249115,0.396076
2,0.634303,0.634720,0.727449,0.724908
3,0.381149,0.348090,0.419633,0.426265
